# Amuzgo Dataset Analysis
This notebook creates the 3 needed datasets without lemma overlap.

In [3]:
# Step 1: Invert columns
input_file = 'azg_original'
output_file = 'azg'
with open(input_file, 'r', encoding='utf8') as fin, open(output_file, 'w', encoding='utf8') as fout:
    for line in fin:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            lemma, form, msd = parts
            fout.write(f'{lemma}\t{msd}\t{form}\n')
print(f'Inverted columns and saved to {output_file}')

Inverted columns and saved to azg


In [4]:
# Step 2: Count unique lemmas
unique_lemmas = set()
line_count = 0
with open(output_file, 'r', encoding='utf8') as fout:
    for line in fout:
        parts = line.strip().split('\t')
        lemma, msd, form = parts
        unique_lemmas.add(lemma)
        line_count += 1
print(f'Counted {len(unique_lemmas)} unique lemmas in {output_file}')

Counted 332 unique lemmas in azg


In [5]:
# Step 3: Analyze lemma counts and feasibility of disjoint splits (train=10k, dev=1k, test=1k)
from collections import defaultdict

lemma_counts = defaultdict(int)
with open(output_file, 'r', encoding='utf8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) != 3:
            continue
        lemma, msd, form = parts
        lemma_counts[lemma] += 1

num_lemmas = len(lemma_counts)
num_lines = sum(lemma_counts.values())

print(f"Total lines: {num_lines}")
print(f"Unique lemmas: {num_lemmas}")
print(f"Avg forms per lemma: {num_lines/num_lemmas:.2f}" if num_lemmas else "Avg forms per lemma: n/a")

required = { 'train': 10_000, 'dev': 1_000, 'test': 1_000 }
print("Required sizes:", required)

if num_lines < sum(required.values()):
    print("Not enough total lines to satisfy 10k/1k/1k even before lemma partitioning.")
else:
    print("Total lines are sufficient in aggregate; checking disjoint-lemma feasibility...")


Total lines: 12204
Unique lemmas: 332
Avg forms per lemma: 36.76
Required sizes: {'train': 10000, 'dev': 1000, 'test': 1000}
Total lines are sufficient in aggregate; checking disjoint-lemma feasibility...


In [6]:
# Step 4: Greedy lemma allocation to prove feasibility (no lemma overlap)
# Strategy:
#  - Fill dev and test first with lemmas of smallest counts (to minimize waste),
#  - Then fill train with remaining lemmas of largest counts,
#  - We can always downsample within a split (keep subset of forms) to hit exact sizes.

required_train, required_dev, required_test = 10_000, 1_000, 1_000

# Sort lemmas by count ascending for dev/test
lemmas_by_count_asc = sorted(lemma_counts.items(), key=lambda kv: kv[1])
lemmas_by_count_desc = sorted(lemma_counts.items(), key=lambda kv: kv[1], reverse=True)

assigned_dev, assigned_test, assigned_train = set(), set(), set()
size_dev = size_test = size_train = 0

# Fill dev
for lemma, cnt in lemmas_by_count_asc:
    if size_dev >= required_dev:
        break
    if lemma in assigned_dev or lemma in assigned_test or lemma in assigned_train:
        continue
    assigned_dev.add(lemma)
    size_dev += cnt

# Fill test (continue scanning from smallest)
for lemma, cnt in lemmas_by_count_asc:
    if size_test >= required_test:
        break
    if lemma in assigned_dev or lemma in assigned_test or lemma in assigned_train:
        continue
    assigned_test.add(lemma)
    size_test += cnt

# Fill train with largest counts from remaining
for lemma, cnt in lemmas_by_count_desc:
    if size_train >= required_train:
        break
    if lemma in assigned_dev or lemma in assigned_test or lemma in assigned_train:
        continue
    assigned_train.add(lemma)
    size_train += cnt

# Results
possible = (size_dev >= required_dev) and (size_test >= required_test) and (size_train >= required_train)

print("Feasibility (>= sizes with disjoint lemmas):", possible)
print(f" - Dev:  {size_dev} lines from {len(assigned_dev)} lemmas (needed {required_dev})")
print(f" - Test: {size_test} lines from {len(assigned_test)} lemmas (needed {required_test})")
print(f" - Train:{size_train} lines from {len(assigned_train)} lemmas (needed {required_train})")

# Sanity checks: no overlap across lemma sets
overlap_dt = assigned_dev & assigned_test
overlap_tt = assigned_test & assigned_train
overlap_td = assigned_train & assigned_dev
print("Overlaps (should be empty):",
      f"dev∩test={len(overlap_dt)}",
      f"test∩train={len(overlap_tt)}",
      f"train∩dev={len(overlap_td)}")

# If not possible, hint at shortfall
if not possible:
    need_dev = max(0, required_dev - size_dev)
    need_test = max(0, required_test - size_test)
    need_train = max(0, required_train - size_train)
    print(f"Shortfall -> dev: {need_dev}, test: {need_test}, train: {need_train}")


Feasibility (>= sizes with disjoint lemmas): True
 - Dev:  1008 lines from 28 lemmas (needed 1000)
 - Test: 1008 lines from 28 lemmas (needed 1000)
 - Train:10008 lines from 271 lemmas (needed 10000)
Overlaps (should be empty): dev∩test=0 test∩train=0 train∩dev=0


In [7]:
# Step 5: Create deterministic disjoint splits and write azg.trn / azg.dev / azg.tst
# Determinism policy:
#  - Tie-breaking by (count, lemma) for lemma selection,
#  - Within each lemma, keep original file order of forms,
#  - Split lemma sets: smallest counts -> dev, then test; largest -> train,
#  - Downsample within each split by truncation to the exact target size.

import os
from collections import defaultdict

required_train, required_dev, required_test = 10_000, 1_000, 1_000

# Build mapping lemma -> list of (lemma, msd, form) in original order
lines_by_lemma = defaultdict(list)
with open(output_file, 'r', encoding='utf8') as f:
    for raw in f:
        parts = raw.rstrip('\n').split('\t')
        if len(parts) != 3:
            continue
        lemma, msd, form = parts
        lines_by_lemma[lemma].append((lemma, msd, form))

# Deterministic counts
counts = [(lem, len(v)) for lem, v in lines_by_lemma.items()]
lemmas_by_count_asc = sorted(counts, key=lambda kv: (kv[1], kv[0]))
lemmas_by_count_desc = sorted(counts, key=lambda kv: (-kv[1], kv[0]))

assigned_dev, assigned_test, assigned_train = set(), set(), set()
size_dev = size_test = size_train = 0

# Fill dev
for lemma, cnt in lemmas_by_count_asc:
    if size_dev >= required_dev:
        break
    if lemma in assigned_dev or lemma in assigned_test or lemma in assigned_train:
        continue
    assigned_dev.add(lemma)
    size_dev += cnt

# Fill test
for lemma, cnt in lemmas_by_count_asc:
    if size_test >= required_test:
        break
    if lemma in assigned_dev or lemma in assigned_test or lemma in assigned_train:
        continue
    assigned_test.add(lemma)
    size_test += cnt

# Fill train
for lemma, cnt in lemmas_by_count_desc:
    if size_train >= required_train:
        break
    if lemma in assigned_dev or lemma in assigned_test or lemma in assigned_train:
        continue
    assigned_train.add(lemma)
    size_train += cnt

# Collect lines deterministically and truncate to match exact sizes
train_lines, dev_lines, test_lines = [], [], []
for lemma in sorted(assigned_dev):
    for trip in lines_by_lemma[lemma]:
        if len(dev_lines) >= required_dev:
            break
        dev_lines.append(trip)
for lemma in sorted(assigned_test):
    for trip in lines_by_lemma[lemma]:
        if len(test_lines) >= required_test:
            break
        test_lines.append(trip)
for lemma in sorted(assigned_train):
    for trip in lines_by_lemma[lemma]:
        if len(train_lines) >= required_train:
            break
        train_lines.append(trip)

# Final sizes
print("Final split sizes (exact):",
      f"train={len(train_lines)}",
      f"dev={len(dev_lines)}",
      f"test={len(test_lines)}")

# Write files next to output_file
base = os.path.splitext(os.path.basename(output_file))[0] or output_file
trn_path = f"{base}.trn"
dev_path = f"{base}.dev"
tst_path = f"{base}.tst"

with open(trn_path, 'w', encoding='utf8') as ftr:
    for lemma, msd, form in train_lines:
        ftr.write(f"{lemma}\t{msd}\t{form}\n")
with open(dev_path, 'w', encoding='utf8') as fdev:
    for lemma, msd, form in dev_lines:
        fdev.write(f"{lemma}\t{msd}\t{form}\n")
with open(tst_path, 'w', encoding='utf8') as ftst:
    for lemma, msd, form in test_lines:
        ftst.write(f"{lemma}\t{msd}\t{form}\n")

print("Wrote:")
print(" -", trn_path, len(train_lines))
print(" -", dev_path, len(dev_lines))
print(" -", tst_path, len(test_lines))

# Verify disjointness
lem_tr = {l for l,_,_ in train_lines}
lem_de = {l for l,_,_ in dev_lines}
lem_te = {l for l,_,_ in test_lines}
print("Lemma overlaps (should be zero):",
      f"train∩dev={len(lem_tr & lem_de)}",
      f"train∩test={len(lem_tr & lem_te)}",
      f"dev∩test={len(lem_de & lem_te)}")


Final split sizes (exact): train=10000 dev=1000 test=1000
Wrote:
 - azg.trn 10000
 - azg.dev 1000
 - azg.tst 1000
Lemma overlaps (should be zero): train∩dev=0 train∩test=0 dev∩test=0
